<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook C01: Feature Engineering</h2>
</div>

Worked solutions to the 4 exercises in
[Notebook C01: Feature Engineering](../notebooks/C01_Feature_engineering.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The feature builders and the model from the notebook.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)

HOLDOUT_DAYS = 90


def add_lags(features, target, lags):
    for lag in lags:
        features[f"lag_{lag}"] = target.shift(lag)
    return features


def add_rolling(features, target, windows, shift=1):
    history = target.shift(shift)
    for window in windows:
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()
        features[f"roll_max_{window}"] = history.rolling(window).max()
    return features


def add_calendar(features, index):
    features["day_of_week"] = index.dayofweek
    features["day_of_month"] = index.day
    features["month"] = index.month
    features["week_of_year"] = index.isocalendar().week.astype(int)
    features["is_weekend"] = (index.dayofweek >= 5).astype(int)
    features["days_since_start"] = (index - index[0]).days
    return features


def add_fourier(features, index, period=365.25, harmonics=2):
    position = (index.dayofyear if period > 100 else index.dayofweek) / period
    for k in range(1, harmonics + 1):
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)
    return features


def build_full_features(target, store):
    features = pd.DataFrame(index=target.index)
    add_lags(features, target, [1, 2, 7, 14, 28])
    add_rolling(features, target, [7, 28])
    add_calendar(features, target.index)
    add_fourier(features, target.index)
    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]
    return features


def holdout_score(features, target=target, holdout=HOLDOUT_DAYS):
    """Fit on everything but the last `holdout` days, score on those."""
    complete = features.notna().all(axis=1)
    X, y = features[complete], target[complete]
    split = len(X) - holdout

    model = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[:split], y.iloc[:split])
    return mean_absolute_error(y.iloc[split:], model.predict(X.iloc[split:]))


features = build_full_features(target, store)
print(f"{features.shape[1]} features, holdout MAE {holdout_score(features):.1f}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Add a **seasonal rolling** feature: the mean of the same weekday over the previous four weeks (`target.shift(7).rolling(4)` applied on a weekday basis, or `target.shift(7).rolling(28).mean()` as an approximation). Does it correlate more strongly with the target than the plain 7-day mean?

In [ ]:
def same_weekday_mean(target, weeks=4):
    """Mean of the same weekday over the previous `weeks` weeks.

    groupby(dayofweek) splits the series into seven weekday-specific series, so
    rolling(4) inside the group steps back four weeks rather than four days. The
    shift(7) comes first, so no row can see its own value.
    """
    return (
        target.shift(7)
        .groupby(target.index.dayofweek)
        .transform(lambda weekday: weekday.rolling(weeks).mean())
    )


candidates = pd.DataFrame({
    "roll_mean_7": target.shift(1).rolling(7).mean(),
    "shift7_roll28": target.shift(7).rolling(28).mean(),
    "same_weekday_4": same_weekday_mean(target),
    "target": target,
}).dropna()

correlations = pd.DataFrame({
    "all days": candidates.corr()["target"],
    "open days only": candidates[store["Open"].reindex(candidates.index) == 1].corr()["target"],
}).drop("target")

correlations.round(3)

**Overall, yes, and dramatically: 0.78 against 0.09. On open days only, no: 0.30 against 0.37.**

The two columns of that table tell opposite stories, and the reason is the reason this exercise is worth
doing.

Store 1 is **closed on Sundays**, and a closed day records zero sales. The same-weekday mean carries that
information perfectly: the previous four Sundays were all zero, so it predicts zero, and it is right. The
plain 7-day mean averages across the week and predicts something like 4,000 on a day the store will take
nothing.

So most of that 0.78 is not seasonal skill. It is the feature having learned to say "Sunday". Restrict to
the days the shop is actually open, where the forecast has to distinguish 4,000 from 5,000 rather than
4,000 from 0, and the advantage reverses.

In [ ]:
comparison = pd.Series({
    "baseline": holdout_score(features),
    "+ same_weekday_4": holdout_score(features.assign(same_weekday_4=same_weekday_mean(target))),
    "+ shift7_roll28": holdout_score(
        features.assign(shift7_roll28=target.shift(7).rolling(28).mean())
    ),
})

print("Holdout MAE")
print(comparison.round(1).to_string())

**And when you put it in the model, it changes nothing at all**: 251.1 becomes 251.8, which is slightly
worse and well inside the noise.

That is the second half of the lesson. The feature is not useless — it genuinely encodes the weekly cycle —
but the model already had `day_of_week`, `lag_7` and `roll_mean_7`, so the weekly cycle was already
represented three times over. A fourth encoding of information the model has adds nothing to predict with
and one more column to split on.

Two things worth taking away:

- **Correlation with the target is a poor guide to whether a feature is worth adding.** It measures the
  feature alone against the target; what matters is what the feature adds *given everything else*.
  Permutation importance, used at the end of the notebook, measures the thing you actually care about.
- **A high correlation driven by a structural zero is not a signal, it is a mask.** Whenever a series
  contains a distinct regime — closed days, a shutdown, a product not yet launched — check your summary
  statistics inside and outside it. Aggregates computed across both mostly describe the difference between
  them.

Note also that the suggested approximation, `target.shift(7).rolling(28).mean()`, is **not** an
approximation of the same-weekday mean: a 28-day window averages over all seven weekdays and so smooths the
weekly cycle away instead of capturing it. It correlates at 0.09, like the plain rolling mean, not at 0.78.
If you want a weekday-specific summary, you have to group by weekday.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Add `target.shift(-1)` (tomorrow's sales) as a feature and evaluate, then do the same with `target` itself. One of the two barely moves the score and the other makes it extraordinary — decide which before you run it. Then check how much of the model each leaking column accounts for, and convince yourself why no amount of cross-validation would have caught either one.

In [ ]:
def score_three_ways(features, target=target, holdout=HOLDOUT_DAYS):
    """Shuffled CV, ordered CV and a future holdout, on the same features."""
    complete = features.notna().all(axis=1)
    X, y = features[complete], target[complete]
    split = len(X) - holdout
    model = HistGradientBoostingRegressor(random_state=0)

    shuffled = -cross_val_score(
        model, X, y, cv=KFold(5, shuffle=True, random_state=0),
        scoring="neg_mean_absolute_error",
    ).mean()
    ordered = -cross_val_score(
        model, X, y, cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error"
    ).mean()

    fitted = model.fit(X.iloc[:split], y.iloc[:split])
    holdout_mae = mean_absolute_error(y.iloc[split:], fitted.predict(X.iloc[split:]))

    return {"Shuffled K-fold": shuffled, "TimeSeriesSplit": ordered, "Future holdout": holdout_mae}


leaks = pd.DataFrame({
    "no leak": score_three_ways(features),
    "+ tomorrow": score_three_ways(features.assign(tomorrow=target.shift(-1))),
    "+ today": score_three_ways(features.assign(today=target)),
})

leaks.round(1)

**`tomorrow` barely registers — 251.1 to 247.9 — while `today` collapses the error to 8.1.**

If you expected tomorrow's sales to be the devastating one, the reason it is not is worth pausing on.
Tomorrow's value only helps to the extent that consecutive days are related, and for this store they barely
are: on open days the correlation between today's sales and tomorrow's is **0.19**. The weekly cycle
dominates the series, and the model already has that. So `target.shift(-1)` is an unambiguous leak — that
column cannot exist at prediction time, ever — and it buys almost nothing.

**That is the more unsettling half of the result.** A leak does not have to announce itself with an
implausible score. This one is undetectable from the metrics: 247.9 is exactly the kind of small
improvement you would report as a modest win and move on. Had you added it by writing `shift(-1)` where you
meant `shift(1)`, nothing in any evaluation would have told you.

`today` is the version with teeth, and it is teaching the same lesson from the other end.

In [ ]:
leaky = features.assign(today=target)
complete = leaky.notna().all(axis=1)
X, y = leaky[complete], target[complete]
split = len(X) - HOLDOUT_DAYS

model = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[:split], y.iloc[:split])

importance = permutation_importance(
    model, X.iloc[split:], y.iloc[split:],
    n_repeats=10, random_state=0, scoring="neg_mean_absolute_error",
)
ranked = pd.Series(importance.importances_mean, index=X.columns).sort_values(ascending=False)

print(ranked.head(5).round(1).to_string())
print()
print(f"'today' accounts for {ranked['today'] / ranked[ranked > 0].sum():.1%} "
      f"of all positive importance")

**`today` and `open` between them account for essentially the whole model**, 53% and 46% of all positive
importance, and everything else has collapsed to noise: `day_of_week` was worth 33 in the honest model and
is worth 2.8 here. Shuffling `today` alone costs 1,005 MAE — four times the entire honest model's error.

Now the part the exercise asks you to convince yourself of. Look back at the three rows of the first table
for the `+ today` column:

| | shuffled K-fold | TimeSeriesSplit | future holdout |
|---|---|---|---|
| **+ today** | 29.5 | 56.2 | 8.1 |

**Every single one is fooled.** Not just the shuffled split that Notebook
[C03](../notebooks/C03_Ensembles.ipynb) and section 3 warn about — the ordered `TimeSeriesSplit`, which
trains only on the past and tests only on the future, reports 56.2 against an honest 398.5. The strictly
chronological future holdout, the most conservative evaluation in the notebook, reports 8.1.

The reason is structural, and it is the single most important idea in this notebook:

> **Cross-validation partitions rows. This leak lives inside a row.**

Every splitting scheme ever devised — shuffled, ordered, grouped, nested, blocked with a purge and an
embargo — decides which rows go in the training set and which go in the test set. Not one of them inspects
what is *in* a row. If the answer is sitting in a column, it travels into the test fold with the row that
carries it, and the test fold reports a triumph.

So splitting strategy protects against one specific failure: information flowing **between** rows, from a
test row's future into a training row. That is a real and common failure, and `TimeSeriesSplit` is the
right tool for it. It offers no protection whatsoever against information flowing **into** a row from its
own future, and that failure is just as common: a centred rolling window, a `shift` with the wrong sign, a
column joined from a table that was itself built later, a status field updated after the fact.

There is only one defence, and it is not a validation scheme. It is asking, of every column: **on the
morning of the day I am forecasting, would this number exist?** The `leakage-proof` cell in the notebook
does exactly that for the centred window, by name and date. It is a tedious question and it has to be
asked column by column, but nothing else will answer it for you.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Train the model from section 5 on `log(sales)` for open days only, predict, and transform back with `exp`. Compare the MAE against the untransformed model, and check whether the back-transformed forecasts are biased low as the note above predicts.

In [ ]:
is_open = store["Open"].reindex(features.index) == 1
complete = features.notna().all(axis=1) & is_open

X, y = features[complete], target[complete]
split = len(X) - HOLDOUT_DAYS
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, actual = y.iloc[:split], y.iloc[split:]

plain = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
logged = HistGradientBoostingRegressor(random_state=0).fit(X_train, np.log(y_train))

predictions = pd.DataFrame({
    "untransformed": plain.predict(X_test),
    "log then exp": np.exp(logged.predict(X_test)),
}, index=actual.index)

summary = pd.DataFrame({
    "MAE": predictions.apply(lambda p: mean_absolute_error(actual, p)),
    "mean error": predictions.apply(lambda p: (p - actual).mean()),
    "median error": predictions.apply(lambda p: (p - actual).median()),
    "% of days over-predicted": predictions.apply(lambda p: (p > actual).mean() * 100),
})

print(f"{len(X)} open days, {HOLDOUT_DAYS} in the holdout\n")
print(summary.round(1).to_string())

**The MAE is unchanged — 298.2 against 298.3 — and the bias question has a more interesting answer than
a yes or a no.**

Both models over-predict, by a lot: +137 and +114 on an average sale of 4,468. So the back-transformed
forecast is not biased low in absolute terms. But **it is biased low relative to the untransformed model**,
by about 23 units, which is the direction the note predicts. The retransformation bias is real and it is
there; it is simply swamped by something larger.

In [ ]:
train_mean, test_mean = y_train.mean(), actual.mean()

print(f"Mean open-day sales, training period: {train_mean:.0f}")
print(f"Mean open-day sales, holdout period:  {test_mean:.0f}")
print(f"Level shift:                          {test_mean - train_mean:+.0f} "
      f"({test_mean / train_mean - 1:+.1%})")
print(f"Holdout runs {actual.index[0].date()} to {actual.index[-1].date()}")

# The theoretical retransformation factor, exp(sigma^2 / 2), needs an honest
# residual spread: a boosted tree's in-sample residuals are far too small.
inner = len(X_train) - HOLDOUT_DAYS
validation_model = HistGradientBoostingRegressor(random_state=0).fit(
    X_train.iloc[:inner], np.log(y_train.iloc[:inner])
)
residuals = np.log(y_train.iloc[inner:]) - validation_model.predict(X_train.iloc[inner:])

print()
print(f"Residual sd on the log scale (validation): {residuals.std():.3f}")
print(f"Implied retransformation factor exp(s²/2): {np.exp(residuals.var() / 2):.4f} "
      f"-> forecasts low by {(np.exp(residuals.var() / 2) - 1) * train_mean:.0f} units")

**The holdout period simply sold less than the training period** — 4,468 against 4,789, down 6.7% — and
both models, having learned the old level, forecast the old level. That level shift is worth about 320
units of error. The retransformation bias is worth about 60. One is five times the other, and the two
happen to point in opposite directions, which is why the log model's over-prediction is the smaller of the
two.

Three conclusions, in descending order of how often they are forgotten:

1. **The transform did not help.** The notebook's own diagnostics said the skew falls from 2.4 to 0.6, so
   the transform does what it claims on the distribution. But a boosted tree is invariant to monotone
   transforms of the *features* and near-indifferent to them on the target when the fit is this good: the
   log scale changes which errors the loss weights most, and here that reweighting is worth nothing. For a
   **linear** model on this data it would matter considerably more. Transform because the model needs it,
   not because the histogram is skewed.

2. **The retransformation bias is real but second-order here**, because `exp(σ²/2)` with σ = 0.15 is 1.012.
   The bias scales with the *square* of the residual spread, so it is negligible for an accurate model and
   serious for a noisy one. A model with σ = 0.5 would be low by 13%, which is the case the note is warning
   about.

3. **Correcting it is not automatic.** Applying a smearing factor estimated from the validation window
   actually makes the holdout MAE worse here (398 against 298), because that window was itself unusual. A
   bias correction estimated on a short, non-representative stretch can easily cost more than the bias.

And the one that outranks all three: **when a forecast is off by 137, look for a level shift before you
look for a retransformation bias.** The distributional subtlety is the interesting explanation; the store
selling less this quarter is the true one.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-4">Exercise 4</h3>
</div>

> Drop every feature whose permutation importance is below 10 and refit. How much accuracy do you lose? Try a gentler threshold of 1 as well. Does either pruned model beat the full one, and does dropping features actually buy back any rows at the start of the series?

In [ ]:
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]
split = len(X) - HOLDOUT_DAYS

model = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[:split], y.iloc[:split])
importance = permutation_importance(
    model, X.iloc[split:], y.iloc[split:],
    n_repeats=15, random_state=0, scoring="neg_mean_absolute_error",
)
ranked = pd.Series(importance.importances_mean, index=X.columns).sort_values(ascending=False)

print(ranked.round(1).to_string())

In [ ]:
def first_usable_row(columns):
    """Where the series starts once only these columns have to be complete."""
    subset = features[columns]
    return subset[subset.notna().all(axis=1)].index[0]


def prune(threshold):
    columns = ranked[ranked >= threshold].index.tolist()
    subset = features[columns]
    usable = subset.notna().all(axis=1)
    return {
        "features": len(columns),
        "usable rows": int(usable.sum()),
        "starts": first_usable_row(columns).date(),
        "holdout MAE": round(holdout_score(subset), 1),
    }


rows = {
    "full model": {
        "features": X.shape[1], "usable rows": len(X),
        "starts": X.index[0].date(), "holdout MAE": round(holdout_score(features), 1),
    },
    "threshold 1": prune(1),
    "threshold 10": prune(10),
}

pd.DataFrame(rows).T

**The aggressive prune costs 61 MAE — 251.1 becomes 312.1, a quarter worse — while the gentle one
*improves* on the full model, 242.5. And neither buys back a single row.**

Taking the two questions in turn.

**On accuracy.** Cutting at 10 leaves seven features and loses badly. Cutting at 1 leaves sixteen and beats
the full model, because the eight it removes measure at or below zero, and five of them are
outright negative: shuffling `roll_std_28`, `fourier_sin_1` or `roll_max_7` *improves* the score. Those are
columns the model is fitting noise on.

The gap between the two thresholds is the whole difficulty of feature selection. Importance near zero means
"contributes nothing measurable **in the presence of the others**", which is not the same as "contributes
nothing". `lag_7` scores 5.7 not because the weekly cycle is unimportant — it is the strongest pattern in
the series — but because `day_of_week` and `lag_1` already carry it. Drop it while keeping them and you
lose little. Drop several such features at once and you can lose the pattern entirely, which is what
happens at threshold 10.

**On rows, the premise does not hold: `lag_28` survives both cuts** with an importance of 14.0, so the
longest lookback is unchanged and the model still starts on 2013-01-29. It is a genuinely useful feature
— roughly a four-week-ago anchor, which for retail is a monthly rhythm — and it is not what you would drop.

Out of curiosity, forcing it out anyway frees 27 rows (941 against 914, starting 2013-01-02) and costs
another 46 MAE, 358.2 in total. **Twenty-seven extra rows at three per cent of the sample, in exchange for
the model's sixth-most-useful feature.** On a longer series the trade would be even more lopsided, since
the rows a lag costs you are a fixed number while its value does not shrink.

Two final cautions, because the threshold-1 result looks better than it is:

- The importances were measured **on the holdout**, and then the pruned models were scored on that same
  holdout. That is selection on the test set, and it biases the comparison in the pruned model's favour. On
  a real project, measure importance on a validation split and keep the holdout untouched. Note that even
  with this advantage, the aggressive prune still lost.
- An 8-point MAE difference on 90 days is not a decisive result. Notebook
  [A06](../notebooks/A06_Evaluating_models.ipynb) showed how much scores move between origins; this
  difference is comfortably inside that range.

The defensible summary is narrower than "pruning helps": **features with reliably negative importance are
worth removing, features with small positive importance are usually worth keeping, and the row cost of a
long lag is rarely the thing that should decide it.**

---

Back to [Notebook C01](../notebooks/C01_Feature_engineering.ipynb), or on to
[Notebook C02](../notebooks/C02_Machine_learning_models.ipynb).